## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Principal Component Analysis](images/img_08_pca_reduction.png)

```
               +---------------------------------------------+
               |        PRINCIPAL COMPONENT ANALYSIS         |
               +---------------------------------------------+
               
        Fitur 2 ^                 .  . (PC 1: Varians Terbesar)
                |               .  /
                |            .   /  .
                |         .    /      .
                |       .     /         .
                |      .    /            .
                |     .   /               .  (PC 2: Ortogonal / 90 deg)
                +--------+---------------------> Fitur 1
```


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_dim = pd.read_csv("../datasets/06_dim_reduction_customer_features.csv")
print("Dataset Customer Features dimuat. Total baris:", len(df_dim))
display(df_dim.head())


## ⚙️ 3. Standarisasi Fitur dan Ekstraksi Komponen Utama (PCA)


In [ ]:
feature_cols = [c for c in df_dim.columns if c != 'customer_id']
X_raw = df_dim[feature_cols]

# Standarisasi data (Mean=0, Std=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# Menjalankan PCA untuk seluruh komponen
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

eigenvalues = pca_full.explained_variance_
explained_var_ratio = pca_full.explained_variance_ratio_
cum_var_ratio = np.cumsum(explained_var_ratio)

pca_summary = pd.DataFrame({
    'Principal Component': [f"PC-{i+1}" for i in range(len(feature_cols))],
    'Eigenvalue': eigenvalues,
    'Varians Terjelaskan (%)': explained_var_ratio * 100,
    'Varians Kumulatif (%)': cum_var_ratio * 100,
    'Kriteria Kaiser (Eigen > 1)': ['Pertahankan' if ev >= 1.0 else 'Eliminasi' for ev in eigenvalues]
})

print("=== Ringkasan Dekomposisi Varians PCA ===")
display(pca_summary.round(2))


## 📉 4. Scree Plot dan Kriteria Kaiser


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Scree Plot (Eigenvalues)
axes[0].plot(range(1, len(eigenvalues)+1), eigenvalues, 'bo-', linewidth=2, markersize=8)
axes[0].axhline(1.0, color='red', linestyle='--', label='Kriteria Kaiser (Eigenvalue = 1)')
axes[0].set_title('Scree Plot: Eigenvalues per Komponen', fontweight='bold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Eigenvalue')
axes[0].legend()

# Subplot 2: Varians Kumulatif
axes[1].bar(range(1, len(cum_var_ratio)+1), cum_var_ratio * 100, alpha=0.6, color='teal')
axes[1].step(range(1, len(cum_var_ratio)+1), cum_var_ratio * 100, where='mid', color='navy', lw=2)
axes[1].axhline(80.0, color='crimson', linestyle=':', label='Target 80% Varians')
axes[1].set_title('Varians Kumulatif Terjelaskan (%)', fontweight='bold')
axes[1].set_xlabel('Jumlah Principal Component')
axes[1].set_ylabel('Varians Kumulatif (%)')
axes[1].legend()

plt.tight_layout()
plt.show()


## 🗺️ 5. Visualisasi 2D PCA Biplot & Heatmap Loading Faktor


In [ ]:
# Menjalankan PCA 2 Komponen Terpilih
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_scaled)
loadings = pd.DataFrame(pca_2d.components_.T, columns=['PC-1 (Aktivitas Digital)', 'PC-2 (Moneter/Belanja)'], index=feature_cols)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Loading Matrix Heatmap
sns.heatmap(loadings, annot=True, fmt=".2f", cmap="coolwarm", cbar=True, ax=axes[0])
axes[0].set_title('Matriks Loading Fitur pada Komponen Utama', fontweight='bold')

# Subplot 2: Proyeksi Ruang 2D PCA
sns.scatterplot(x=X_pca_2d[:, 0], y=X_pca_2d[:, 1], ax=axes[1], color='darkcyan', alpha=0.8, s=60)
axes[1].set_title('Proyeksi Pengguna pada Ruang 2D PCA', fontweight='bold')
axes[1].set_xlabel(f'PC-1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% Varians)')
axes[1].set_ylabel(f'PC-2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% Varians)')

plt.tight_layout()
plt.show()


## 📝 Kesimpulan Analisis

### Q&A
* **Berapa banyak komponen utama yang harus dipertahankan?** Berdasarkan kriteria Kaiser (*Eigenvalue* $\ge 1.0$) dan *Scree Plot*, **2 komponen pertama** cukup dipertahankan karena berhasil menangkap $>75\%$ total varians data awal.

### Data Analysis Key Findings
* Komponen 1 (PC-1) mewakili dimensi **"Intensitas Aktivitas Digital"** (memiliki bobot loading tinggi pada `login_frequency`, `browsing_duration`, dan `search_queries`).
* Komponen 2 (PC-2) mewakili dimensi **"Kapasitas Belanja / Nilai Moneter"** (memiliki bobot loading tinggi pada `avg_cart_value` dan `customer_lifetime_value`).

### Insights or Next Steps
* Dataset 8 fitur awal berhasil dikompresi menjadi 2 fitur laten murni tanpa kehilangan informasi signifikan, siap digunakan untuk klasterisasi cepat pada Modul 09.
